# Comparing proportions 
## Introduction


## Fisher's exact test
### When to use Fisher's test
### Contingency table


In [ ]:
import numpy as np
from scipy.stats import contingency

# Example data
treatment = np.array([1, 1, 2, 2, 1, 2, 1]) # 1 = Drug A, 2 = Drug B
outcome   = np.array([0, 1, 0, 1, 1, 1, 0]) # 0 = No response, 1 = Response

# Create contingency table withSciPy's stats.contingency
# with treatment as rows, outcome as columns
res = contingency.crosstab(treatment, outcome)

# If data is already in a DataFrame we can use pandas.crosstab
import pandas as pd

table_df = pd.DataFrame({"treatment": treatment, "outcome": outcome})

# Creating contingency table using pd.crosstab
table_crosstab = pd.crosstab(
    table_df["treatment"].dropna(),  # Remove NaN values
    table_df["outcome"],
    margins=True,              # Include row and column totals
    dropna=False               # Keep rows/columns with zero counts!
)

print("Using `scipy.stats.contingency`")
print("------------------------------")
print(f"The object containing all atributes is:\n{res}")
print(f"The `treatment` (rows) elements are: {res.elements[0]}")
print(f"The `outcome` (columns) elements are: {res.elements[1]}")
print(f"The actual contingency table is:\n{res.count}")
print()
print("Using `pandas.crosstab`")
print("----------------------")
print(table_crosstab)

In [ ]:
# Contingency table with rows = alternative treatments, 
# cols = alternative outcomes
# table = np.array([[73, 756], [14, 826]])

table_apixaban_df = pd.DataFrame(
    {   # Pre-aggregated data
        'recurrence':    [73, 14],
        'no_recurrence': [756, 826],
    },
    index=['placebo', 'apixaban'])

print(table_apixaban_df)

### P value
#### Hypergeometric distribution


In [ ]:
from scipy.stats import hypergeom

# Define the parameters for the hypergeometric distribution:
M = 40  # Total population size (grand total)
n = 29  # Number of white balls in the population
N = 20  # Number of draws (sample size)
k = 18  # Number of white balls in the sample drawn

# Calculate the probability mass function (PMF) for k successes
hypergeometric_dist = hypergeom(M=M, n=n, N=N)
pmf_hypergeom = hypergeometric_dist.pmf(k=k)

# Print the result with formatted output, displaying 5 significant digits.
print(f"Hypergeometric distribution with {M=}; {N=}; {n=}:")
print(f" P(K=18) = {pmf_hypergeom:.5}")

#### Relationship between hypergeometric distribution and P value


In [ ]:
from scipy.stats import fisher_exact

# Contingency table using NumPy array
table = np.array([[18, 2], [11, 9]])

# Perform Fisher's exact test
odds_ratio, p_value = fisher_exact(table, alternative='two-sided')
print(f"Fisher's exact test P value (two-sided); SciPy)={p_value:.4f}")

# Parameters for the hypergeometric distribution (from contingency table)
M = np.sum(table)             # Sum all cells
n = np.sum(table, axis=0)[0]  # Sum of the first column (cases)
N = np.sum(table, axis=1)[0]  # Sum of the first row (control)

# Calculate one-sided P value
k = table[0, 0]  # Number of successes (cases in control)
p_value_one_sided = hypergeom.sf(k=k - 1, M=M, n=n, N=N)
print("Hypergeometric distribution:")
print(f"  One-sided P value (>= {k} 'successes')={p_value_one_sided:.4f}")

# Calculate two-sided P value (approximation)
# We simply multiply the one-sided P value by 2
print(f"  Two-sided P value={2 * p_value_one_sided:.4f}")

In [ ]:
# Probabilities of individual tables
print("Probabilities of individual tables (more extreme than observed):")

# Consider tables with more extreme 'successes'
extreme_k_values = [9, 10, 11, 18, 19, 20]
total_probability = 0  # Initialize the sum of probabilities

for k in extreme_k_values:
    prob = hypergeometric_dist.pmf(k=k)
    print(f"  k = {k}:\t{prob:.4f}")
    total_probability += prob  # Accumulate the probability

print(f"Sum of probabilities (more extreme): {total_probability:.4f}")

#### Interpreting the P value


In [ ]:
# For pre-aggregated data we can converts DataFrame to NumPy array
table_apixaban_array = table_apixaban_df.to_numpy()
odds_ratio_fisher, p_value_fisher = fisher_exact(table_apixaban_array)

print(f"P value in the apixaban study = {p_value_fisher:.3e}")

#### One-sided test - application to toxicology count data


In [ ]:
# Contingency table: Tumor counts in treated vs. control groups
tumor_table = pd.DataFrame(
    data={'treated': [8, 42], 'control': [2, 48]},
    index=['tumor', 'no_tumor'])

print("Tumor count table:")
print(tumor_table)

# Perform one-sided Fisher's exact test (alternative='greater')
odds_ratio_tumors, p_value_tumors = fisher_exact(
    tumor_table, alternative='greater')

# Display results
print()
print(f"Odds ratio: {odds_ratio_tumors:.2f}")
print(f"P value (one-sided): {p_value_tumors:.3f}")


### Calculating and interpreting risk metrics
#### Absolute risk reduction and attributable risk


In [ ]:
def calculate_arr(table):
    """Calculates absolute risk reduction from a 2×2 contingency table.

    Args:
        table: a 2×2 NumPy array representing the contingency table.
            Rows: control or exposed/treatment; columns: disease/no disease

    Returns:
        float: the absolute risk reduction.
    """

    # Check if table is 2×2
    if table.shape != (2, 2):
        raise ValueError("Input table must be a 2×2 NumPy array.")

    # Calculate risk in exposed and unexposed groups
    risk_control =   table[0, 0] / np.sum(table[0, :])
    risk_treatment = table[1, 0] / np.sum(table[1, :])

    # Calculate attributable risk
    ar = risk_control - risk_treatment

    return ar

In [ ]:
# Application to our Apixaban example
arr = calculate_arr(table_apixaban_array)

print(f"Abosulte risk redution (ARR) in apixaban study = {100*arr:.2f}%")

#### Population attributable risk 


In [ ]:
p_exposed   = 150 / (150 + 850)
p_unexposed =  50 / ( 50 + 950)
exposure_prevalence = 0.20

PAR = (p_exposed - p_unexposed) * exposure_prevalence
print(f"Population attributable risk (PAR) = {100*PAR:.2f}%")

#### Number needed to treat


In [ ]:
def calculate_nnt(absolute_risk_reduction):
    """Calculates number needed to treat or number needed to harm.

    Args:
        absolute_risk_reduction: the absolute risk reduction 
        (risk difference or AR).

    Returns:
        int or float: 
            - If ARR > 0: NNT (rounded to nearest integer).
            - If ARR < 0: NNH (rounded to nearest integer).
            - If ARR = 0: None (no effect)
    """

    if absolute_risk_reduction > 0:
        return round(1 / absolute_risk_reduction)  # NNT
    elif absolute_risk_reduction < 0:
        return round(-1 / absolute_risk_reduction) # NNH
    else:
        return None  # No effect

In [ ]:
# Application to our Apixaban example
nnt = calculate_nnt(arr)
print(f"Number needed to treat (NNT) in apixaban study = {nnt}")

#### Relative risk


In [ ]:
def calculate_rr(table_or_p1, p2=None):
    """Calculates relative risk from a contingency table or proportions.

    Args:
        table_or_p1: either a 2×2 NumPy array representing a contingency 
            table, or the proportion (float) of events in the exposed group.
        p2 (float): if table_or_p1 is a proportion, this should be 
            the proportion of events in the unexposed group.

    Returns:
        float: the relative risk, or None if calculation is not possible.
    """

    if p2 is None:
        # Input is a contingency table
        if not isinstance(
            table_or_p1, np.ndarray) or table_or_p1.shape != (2, 2):
            raise ValueError("Invalid input: provide a 2×2 NumPy array.")

        a, b, c, d = table_or_p1.ravel()
        p1 = a / (a + b)
        p2 = c / (c + d)
    else:
        # Input is already proportions
        p1 = table_or_p1

    # Check for division by zero
    if p2 == 0:
        return None

    rr = p1 / p2

    return rr

In [ ]:
# Calculate RR from contingency table
rr_table = calculate_rr(table_apixaban_array)

# Calculate RR using scipy.stats.contingency
a, b, c, d = table_apixaban_array.ravel()
rr_scipy = contingency.relative_risk(
    exposed_cases=a,
    exposed_total=a + b,
    control_cases=c,
    control_total=c + d).relative_risk

# Calculate RR from proportions with a different reference
placebo_cases = table_apixaban_array[0, 0]  # placebo_cases = a
placebo_total = np.sum(table_apixaban_array[0, :])  # placebo_total = a + b
apixaban_cases = table_apixaban_array[1, 0]  # apixaban_cases = c
apixaban_total = np.sum(table_apixaban_array[1, :])  # apixaban_total = c+d

p1_apaxiban = placebo_cases / placebo_total
p2_apaxiban = apixaban_cases / apixaban_total
rr_proportions = calculate_rr(p2_apaxiban, p1_apaxiban)

print("Apixaban study:")
print(f" Relative risk (manual) = {rr_table:.3f}")
print(f" Relative risk (SciPy) = {rr_scipy:.3f}")
print(f" Relative risk (from proportions, reference = placebo) = \
{rr_proportions:.3f}")

#### LogRR


In [ ]:
import math

# rr_table is the previously calculated Relative Risk
log_rr = math.log(rr_table)  # Calculate the natural logarithm of RR

print("Apixaban study:")
print(f" Log(RR) = {log_rr:.3f}")

# Convert LogRR back to RR
print(f" Back transformation to RR: {math.exp(log_rr):.3f}")

#### Odds ratio


In [ ]:
def calculate_or(table):
    """Calculates odds ratio from a 2×2 contingency table.

    Args:
        table: a 2×2 NumPy array representing the contingency table
               in the format [[a, b], [c, d]].

    Returns:
        float: the odds ratio.
    """
    if not isinstance(table, np.ndarray) or table.shape != (2, 2):
        raise ValueError("Invalid input: Must provide a 2×2 NumPy array.")

    a, b, c, d = table.ravel()
    return (a * d) / (b * c)

In [ ]:
# 1Calculate OR from custom function
oddsratio_custom = calculate_or(table_apixaban_array)

# Calculate OR from scipy.stats.contingency
oddsratio_scipy = contingency.odds_ratio(
    table=table_apixaban_array,  # Use the already converted NumPy array
    kind='sample').statistic

print("Apixaban study:")
print(f" Odds ratio (manual) = {oddsratio_custom:.3f}")
print(f" Odds ratio (SciPy) = {oddsratio_scipy:.3f}")

In [ ]:
print(f"Odds ratio (from `fisher_exact`): {odds_ratio_fisher:.3f}")

#### LogOR


In [ ]:
# Log transformation of the OR returned by `fisher_exact`
log_odds_ratio = math.log(odds_ratio_fisher)
print(f"Log(OR) (log of `fisher_exact` OR) = {log_odds_ratio:.3f}")

# log(OR) obtained with calculations and p1 and p2 calculated previously
log_odds_ratio_calc = (
    math.log(p1_apaxiban)
    - math.log(1 - p1_apaxiban)
    - math.log(p2_apaxiban)
    + math.log(1 - p2_apaxiban)
)

print(f"Log(OR) (manual calculation; reference = placebo): \
{log_odds_ratio_calc:.3f}")

### Confidence intervals


#### CI for proportions


In [ ]:
from statsmodels.stats.proportion import proportion_confint
import warnings
warnings.filterwarnings("ignore")  # Silence statsmodels warnings

# Calculate CI for placebo group (using normal approximation)
# alpha=0.05 by default
ci_placebo = np.array(proportion_confint(
    count=placebo_cases, nobs=placebo_total, method='normal'))

# Calculate CI for apixaban group
ci_apixaban = np.array(proportion_confint(
    count=apixaban_cases, nobs=apixaban_total, method='normal'))

print("Apixaban study - Proportion and 95% CI:")

# We round each bound of the returned tuples using a generator
print(
    " placebo group:",
    round(placebo_cases/placebo_total, 3),
    ci_placebo.round(4))
print(
    " apixaban group:",
    round(apixaban_cases/apixaban_total, 3),
    ci_apixaban.round(4))

#### CI for ARR
##### Standard approximation


In [ ]:
from scipy.stats import norm

def calculate_arr_ci(table, conf=0.95):
    """Calculates absolute risk reduction and its confidence interval 
        from a 2×2 contingency table.

    Args:
        table: a 2×2 NumPy array representing the contingency table.
        conf: confidence level (default: 0.95).

    Returns:
        tuple: a tuple containing:
            - ARR: the absolute risk reduction.
            - s_ARR: the standard error of the ARR.
            - p_bar: overall risk.
            - ci_lower: lower bound of the confidence interval.
            - ci_upper: upper bound of the confidence interval.
                We return these two in an np.array
    """
    if table.shape != (2, 2):
        raise ValueError("Input table must be a 2×2 array.")

    a, b, c, d = table.ravel()

    # Risk calculations
    p1 = a / (a + b)
    p2 = c / (c + d)
    arr = p1 - p2

    # Standard error of AR
    M = a + b + c + d   # total number of observations
    n = a + c           # total number of event
    p_bar = n / M # overall risk
    se_arr = np.sqrt(p_bar * (1 - p_bar) * (1/(a+b) + 1/(c+d)))

    # Z-score for desired confidence level
    z_crit = norm.ppf((1 + conf) / 2)

    # Confidence interval (adjust for negative AR if necessary)
    ci_lower = arr - z_crit * se_arr
    ci_upper = arr + z_crit * se_arr

    return arr, se_arr, p_bar, np.array((ci_lower, ci_upper))

In [ ]:
# Application with the apixaban example
arr, se_arr, p_bar, ci_arr = calculate_arr_ci(table_apixaban_array)

print("Apixaban study:")
print(f" ARR = {100*arr:.3f}%")
print(f" Standard error (ARR) = {se_arr:.5f}")
print(f" Overall risk (p) = {p_bar:.3f}")
print(r" 95% CI (ARR):", ci_arr.round(4))

##### Z-test for comparing proportions


In [ ]:
from statsmodels.stats.proportion import proportions_ztest

# Extract counts and total observations 
# for each group from the apixaban table
count = table_apixaban_df['recurrence'] # Gives Series with 2 values
# Series of the total nobs for each group
nobs = table_apixaban_df.sum(axis=1)

# Perform z-test for difference in proportions
z_stat, p_value_z_test = proportions_ztest(count, nobs)

print("Apixaban study:")
print(f" Z-statistic (using previous SE) = \
{(p1_apaxiban - p2_apaxiban) / se_arr:.2f}")
print(f" Z-statistic (statsmodels) = {z_stat:.2f}")
print(f" P value (statmodels) = {p_value_z_test:.4e}")

##### Exact methods


In [ ]:
# Calculate standard error using individual proportions
se_exact = np.sqrt(
    (p1_apaxiban * (1 - p1_apaxiban) / placebo_total)
    + (p2_apaxiban * (1 - p2_apaxiban) / apixaban_total)
)
print("Apixaban study:")
print("Exact standard error (s) =", round(se_exact, 5))

In [ ]:
from statsmodels.stats.proportion import confint_proportions_2indep

ci_arr_newcombe = confint_proportions_2indep(
    count1=placebo_cases,
    nobs1=placebo_total,
    count2=apixaban_cases,
    nobs2=apixaban_total,
    compare='diff',
    method='newcomb',
    alpha=0.05  # Default
)

ci_arr_newcombe = np.array(ci_arr_newcombe)

print("Apixaban study:")
print(r" 95% CI (ARR; Newcombe):", ci_arr_newcombe.round(3))

#### CI for NNT


In [ ]:
def calculate_nnt_ci(table, conf=0.95):
    """
    Calculates NNT (or NNH) and its confidence interval from a 2×2 contingency table.

    Args:
        table: a 2×2 NumPy array representing the contingency table.
        conf: confidence level (default: 0.95).

    Returns:
        tuple: a tuple containing:
            - nnt_or_nnh (float): NNT if ARR > 0, NNH if ARR < 0, 
                None if ARR = 0.
            - ci_lower (float or str): lower bound of CI 
                (or "Infinity" if undefined).
            - ci_upper (float or str): upper bound of CI 
                (or "Infinity" if undefined).
                Returned as a np.array.
    """

    arr, _, _, ci_arr = calculate_arr_ci(table, conf)

    if arr > 0:  # Calculate NNT
        nnt_or_nnh = round(1 / arr)
        ci_lower = round(1 / ci_arr[1])  # Invert CI for NNT
        ci_upper = round(1 / ci_arr[0])
    elif arr < 0:  # Calculate NNH
        nnt_or_nnh = round(-1 / arr)
        ci_lower = round(-1 / ci_arr[0])  # Invert CI for NNH
        ci_upper = round(-1 / ci_arr[1])
    else:
        return None, "Infinity", "Infinity"  # No effect, CI undefined

    # Handle undefined bounds when CI for ARR includes zero
    if ci_arr[0] <= 0:
        ci_lower = "Infinity"
    if ci_arr[1] <= 0:
        ci_upper = "Infinity"

    return nnt_or_nnh, np.array((ci_lower, ci_upper))

In [ ]:
# Application with the apixaban example
nnt_or_nnh, ci_ntt = calculate_nnt_ci(table_apixaban_array)

print("Apixaban study:")

if nnt_or_nnh is not None:
    metric_name = "NNT" if nnt_or_nnh > 0 else "NNH"
    print(f" {metric_name}: {nnt_or_nnh}")
    print(r" 95% CI (NTT):", ci_ntt.round(3))
else:
    print("No treatment effect (ARR = 0)")

#### CI for RR


In [ ]:
def calculate_rr_ci(table, conf=0.95):
    """Calculates relative risk (RR) and its confidence interval (CI) 
    from a 2×2 contingency table.

    Args:
        table: a 2×2 NumPy array representing the contingency table.
        conf: confidence level (default: 0.95).

    Returns:
        tuple: a tuple containing:
            - RR: the relative risk.
            - ci_lower: lower bound of the confidence interval.
            - ci_upper: upper bound of the confidence interval.
                The bounds are returned as a single np.array.
    """

    if table.shape != (2, 2):
        raise ValueError("Input table must be a 2×2 array.")

    a, b, c, d = table.ravel()

    # Risk calculations (AR = p2 - p1)
    p1 = a / (a + b)
    p2 = c / (c + d)
    rr = p1 / p2

    # Standard error of RR
    log_rr = np.log(rr)
    se_log_rr = np.sqrt((1 - p1) / a + (1 - p2) / c)

    # Z-score for desired confidence level
    z = norm.ppf((1 + conf) / 2)

    # Confidence interval on the log scale
    ci_lower_log = log_rr - z * se_log_rr
    ci_upper_log = log_rr + z * se_log_rr
    ci_lower = np.exp(ci_lower_log)
    ci_upper = np.exp(ci_upper_log)

    return rr, np.array((ci_lower, ci_upper))

In [ ]:
# Application with the apixaban example
rr, ci_rr = calculate_rr_ci(table_apixaban_array)
print("Apixaban study:")
print(f" Relative risk (RR) = {rr:.3f}")
print(r" 95% CI (RR):", ci_rr.round(3))

In [ ]:
# Calculate the confidence interval using the score method
ci_rr_log = confint_proportions_2indep(
    count1=placebo_cases,
    nobs1=placebo_total,
    count2=apixaban_cases,
    nobs2=apixaban_total,
    compare='ratio',
    method='log',  # Available ['log', 'log-adjusted', 'score']
)

ci_rr_log = np.array(ci_rr_log)

print("Apixaban study:")
print(r" 95% CI (RR; log):", ci_rr_log.round(3))

#### CI for OR


In [ ]:
def calculate_or_ci(table, conf=0.95):
    """Calculates odds ratio and its confidence interval from 
    a 2×2 contingency table.

    Args:
        table: A 2×2 NumPy array representing the contingency table.
        conf: Confidence level (default: 0.95).

    Returns:
        tuple: A tuple containing:
            - OR: The odds ratio.
            - ci_lower: Lower bound of the confidence interval.
            - ci_upper: Upper bound of the confidence interval.
                The CI is returned as a np.array.
    """

    if table.shape != (2, 2):
        raise ValueError("Input table must be a 2×2 array.")

    a, b, c, d = table.ravel()

    # Odds ratio calculation
    odds_ratio = (a * d) / (b * c)
    log_or = np.log(odds_ratio)

    # Standard error of log odds ratio
    se_log_or = np.sqrt(1/a + 1/b + 1/c + 1/d)

    # Z-score for desired confidence level
    z = norm.ppf((1 + conf) / 2)

    # Confidence interval calculation
    ci_lower = np.exp(log_or - z * se_log_or)
    ci_upper = np.exp(log_or + z * se_log_or)

    return odds_ratio, np.array((ci_lower, ci_upper))

In [ ]:
# Application with the apixaban example
or_val, ci_or = calculate_or_ci(table_apixaban_array)
print("Apixaban study:")
print(f" Odds ratio (OR): {or_val:.3f}")
print(r" 95% CI (OR):", ci_or.round(3))

In [ ]:
# Calculate the confidence interval using the score method
ci_or_score = confint_proportions_2indep(
    count1=placebo_cases,
    nobs1=placebo_total,
    count2=apixaban_cases,
    nobs2=apixaban_total,
    compare='odds-ratio',
    method='score',  # Available ['logit', 'logit-adjusted', 'score']
)

ci_or_score = np.array(ci_or_score)

print("Apixaban study:")
print(r" 95% CI (OR; score):", ci_or_score.round(3))

### Summary table
#### Putting it all together


In [ ]:
# See https://docs.python.org/3/library/typing.html for type hints
# notations sush as -> pd.DataFrame

def calculate_metrics(table, conf=0.95) -> pd.DataFrame:
    """Calculates various metrics and confidence intervals from a 
    2×2 contingency table.

    Args:
        table: a 2×2 NumPy array representing the contingency table.
        conf: confidence level (default: 0.95).

    Returns:
        pd.DataFrame: a DataFrame containing 
            the calculated metrics and CIs.
    """

    a, b, c, d = table.ravel()

    # Proportions
    p1 = a / (a + b)
    p2 = c / (c + d)

    # Confidence intervals for proportions
    ci_p1 = proportion_confint(a, a + b, alpha=1-conf, method='wilson')
    ci_p2 = proportion_confint(c, c + d, alpha=1-conf, method='wilson')

    # Attributable risk and CI
    arr, _, _, ci_arr = calculate_arr_ci(table, conf)

    # Number needed to treat (NNT) or harm (NNH) and CI
    nnt_or_nnh, ci_ntt = calculate_nnt_ci(table, conf)

    # Odds ratio and CI
    odds_ratio, ci_or = calculate_or_ci(table, conf)

    # Relative risk and CI
    rr, ci_rr = calculate_rr_ci(table, conf)

    # Log transformations and respective CIs
    log_or = np.log(odds_ratio)
    log_rr = np.log(rr)
    se_log_or = np.sqrt(1/a + 1/b + 1/c + 1/d)
    se_log_rr = np.sqrt((1 - p1)/a + (1 - p2)/c)

    z = norm.ppf((1 + conf) / 2)

    ci_lower_log_or = log_or - z * se_log_or
    ci_upper_log_or = log_or + z * se_log_or

    ci_lower_log_rr = log_rr - z * se_log_rr
    ci_upper_log_rr = log_rr + z * se_log_rr

    # Fisher's exact test
    _, p_value = fisher_exact(table)


    # Create results DataFrame
    data = {
        'Metric': [
            'P value', 'Proportion (control)', 'Proportion (treatment)',
            'ARR', 'NNT/NNH', 'OR', 'log(OR)', 'RR', 'log(RR)'],
        'Estimate': [
            p_value, p1, p2, arr, nnt_or_nnh, 
            odds_ratio, log_or, rr, log_rr],
        'CI lower': [
            np.nan, ci_p1[0], ci_p2[0], ci_arr[0], ci_ntt[0],
            ci_or[0], ci_lower_log_or, ci_rr[0], ci_lower_log_rr],
        'CI upper': [
            np.nan, ci_p1[1], ci_p2[1], ci_arr[1], ci_ntt[1],
            ci_or[1], ci_upper_log_or, ci_rr[1], ci_upper_log_rr]
    }

    return pd.DataFrame(data)

In [ ]:
summary_df_apixaban = calculate_metrics(table_apixaban_array)
print("Comparisons between proportions in tha apixaban study:\n")
print(
    summary_df_apixaban
    .round(3)  # Round to 3 decimal places
    .to_markdown(index=False, numalign="left", stralign="left"))

#### Table2×2 for risk and odds ratio analysis


In [ ]:
from statsmodels.stats.contingency_tables import Table2x2

# Assuming 'table' is the 2×2 contingency table (NumPy array)
table2x2 = Table2x2(table_apixaban_array)

# Calculate and print the results
print(table2x2.summary())

## Chi-squared test
### How it works


### Example of the Mendel's peas


### χ² goodness-of-fit test


In [ ]:
from scipy.stats import chisquare

# Mendel's observed data
observed_mendel_freq = np.array([315, 108, 101, 32])

# Mendel's expected proportions and frequencies
expected_mendel_prop = np.array([9/16, 3/16, 3/16, 1/16])
expected_mendel_freq = observed_mendel_freq.sum() * expected_mendel_prop

# Perform the chi-squared goodness-of-fit test
chi2_stat, p_value_chi2 = chisquare(
    f_obs=observed_mendel_freq,
    f_exp=expected_mendel_freq)

# Print the results
print("Observed frequencies:", observed_mendel_freq)

print("Expected frequencies:", expected_mendel_freq.round(2))

print("Chi-squared statistic =", chi2_stat.round(2))
print(f"P value = {p_value_chi2:.4f}")

# Interpretation
α = 0.05

if p_value_chi2 < α:
    print(f"Reject the null hypothesis (P < {α}). \
The observed frequencies do not fit the expected proportions.")
else:
    print(f"Fail to reject the null hypothesis.\n\
The observed frequencies are consistent with the expected proportions.")

#### The χ²-distribution


In [ ]:
#| fig-subcap: 
#|   - "Different chi-squared distributions with varying DFs"
#|   - "Comparison of the χ² and normal distributions"
#| layout-ncol: 2

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2

# Degrees of freedom to plot
degrees_of_freedom = [1, 2, 3, 7, 15]

# Use a context manager to change color palette only in this figure
with sns.color_palette("magma_r"):

    # Generate x values for the distribution
    x = np.linspace(0, 30, 200)  # To capture most of the distribution

    # Create the plot
    plt.figure(figsize=(3.5, 3))
    for df in degrees_of_freedom:
        # Calculate chi-squared PDF
        chi2_pdf = chi2(df=df).pdf(x)
        plt.plot(x, chi2_pdf, label=f'DF={df}')

    # Add labels and title
    plt.xlabel('k')
    plt.ylim(0, 0.4)
    plt.yticks([])
    plt.ylabel('Probability density')
    plt.margins(x=0, y=0)
    plt.title('χ² with varying DFs')
    plt.legend()
    plt.show()  # Required only for building figure in the book

    # Comparison of the chi-squared and normal distributions
    # Set up parameters
    DF_max = max(degrees_of_freedom)

    # Calculate PMF and PDF values
    chi2_pdf = chi2(DF_max).pdf(x)
    μ = DF_max
    σ = (2 * DF_max)**.5  # Std is square-root of variance
    norm_pdf = norm(loc=μ, scale=σ).pdf(x)

    # Plot
    plt.figure(figsize=(3.5, 3))
    plt.plot(
        x, chi2_pdf,
        lw=3, ls='--', color='blue',
        label=f"χ² (DF={DF_max})")

    plt.plot(x, norm_pdf, lw=3, label=f"Normal (μ={μ}, σ={σ:.2f})")
    plt.title('χ² and normal distributions')
    plt.xlabel('k or z')
    plt.ylabel('Probability density')
    plt.margins(x=0, y=0)
    plt.yticks([])
    plt.legend()
    plt.show()

#### Degrees of freedom
#### Visualizing the χ² test


In [ ]:

# Degrees of freedom for Mendel's example
DF_mendel = len(observed_mendel_freq) - 1

# Calculate critical value
chi2_crit_goodness = chi2.ppf(1 - α, df=DF_mendel)

# Calculate P value using chi-squared stat from previous calculation
# but we could also take the value returned by `chisquare`
p_value_goodness = 1 - chi2.cdf(chi2_stat, df=DF_mendel)

# Generate x values for plotting
x = np.linspace(0, 15, 1000)
hx = chi2.pdf(x, df=DF_mendel)

# Create the plot
plt.figure(figsize=(3.5, 3))
plt.plot(x, hx, lw=2, color="black")

# Plot the critical value
plt.axvline(
    x=chi2_crit_goodness,
    color='orangered', linestyle='--',
    label=f'χ²* ({chi2_crit_goodness:.2f})')

# Shade the probability alpha
plt.fill_between(
    x[x >= chi2_crit_goodness], hx[x >= chi2_crit_goodness],
    linestyle = "-", linewidth = 2, color = 'tomato',
    label=f'α ({α})')

# Plot the observed chi-squared statistic
plt.axvline(
    x=chi2_stat,
    color='limegreen', linestyle='--',
    label=f'χ² ({chi2_stat:.2f})')

# Shade the P value area
plt.fill_between(
    x[x >= chi2_stat], hx[x >= chi2_stat],
    # hatch='///', edgecolor="limegreen", facecolor='lime', alpha=.5,
    color='greenyellow',
    label=f'P ({p_value_goodness:.2f})')

# Add labels and title
plt.xlabel('χ²')
plt.ylabel('Density')
plt.margins(x=0.01, y=0)
plt.yticks([])
plt.title(f'χ²-distribution (DF={DF_mendel})')

plt.legend();

### χ² test of independence


#### Mosaic plot


In [ ]:

from statsmodels.graphics.mosaicplot import mosaic

# Create a Series from the observed data
observed_mendel_series = pd.Series(
    observed_mendel_freq,
    index=[
        ('Round', 'Yellow'),
        ('Round', 'Green'),
        ('Wrinkled', 'Yellow'),
        ('Wrinkled', 'Green')])

# Create a mosaic plot
_, ax = plt.subplots(nrows=1, ncols=1, figsize=(3.5, 3))

mosaic(observed_mendel_series, ax=ax)
plt.title("Mosaic plot of Mendel's experiment");

#### Interpretation of the P value


In [ ]:
from scipy.stats import chi2_contingency

# Reshaping the previous contingency table, rows will be 
# seed shape (round, wrinkled), columns will be color (yellow, green)
observed_mendel_contingency = observed_mendel_freq.reshape(2, 2)

chi2_independ, pval_independ, \
dof_independ, expected_independ = chi2_contingency(
    observed=observed_mendel_contingency,
    lambda_='pearson',
    # Available 'cressie-read', 'log-likelihood', 'neyman', etc.
    correction=True, # Yates' correction for continuity (default)
)

# Print the results
print("χ² test of independence")
print("-----------------------")
print("Observed Frequencies:")
print(observed_mendel_contingency)
print()
print("Expected Frequencies (under null hypothesis of independence):")
print(expected_independ.round(2))
print()
print(
    "χ²-statistic =",
    chi2_independ.round(4))
print("Degrees of freedom =", dof_independ)
print(f"P value = {pval_independ:.4f}")

# Interpret the results (using alpha = 0.05 as significance level)
alpha = 0.05
if pval_independ < alpha:
    print("\nReject the null hypothesis. There is a significant \
association between seed shape and color.")
else:
    print("\nFail to reject the null hypothesis. \n There is not enough \
evidence to suggest an association between seed shape and color.")

#### Degrees of freedom in tests of independence


#### Yates' correction


#### Visualizing the χ² test of independence


In [ ]:

# Calculate critical value
# dof_independ comes from the previous test chi2_contingency
chi2_crit_indep = chi2.ppf(1 - α, df=dof_independ)

# Calculate P value, but could also use pval from `chi2_contingency`
p_value_indep = 1 - chi2.cdf(chi2_independ, df=dof_independ)

# Generate x values for plotting
x = np.linspace(0, 10, 100)
hx = chi2.pdf(x, df=dof_independ)

# Create the plot
plt.figure(figsize=(3.5, 3))
plt.plot(x, hx, lw=2, color="black")

# Plot the critical value
plt.axvline(
    x=chi2_crit_indep,
    color='orangered',
    linestyle='--',
    label=f'χ²* ({chi2_crit_indep:.2f})')

# Shade the probability alpha
plt.fill_between(
    x[x >= chi2_crit_indep], hx[x >= chi2_crit_indep],
    linestyle = "-", linewidth = 2, color = 'tomato',
    label=f'α ({α})')

# Plot the observed χ² statistic of the test of independence
plt.axvline(
    x=chi2_independ,
    color='limegreen',
    linestyle='--',
    label=f'χ² ({chi2_independ:.3f})')

# Shade the P value area
plt.fill_between(
    x[x >= chi2_independ], hx[x >= chi2_independ],
    color='greenyellow',
    label=f'P ({p_value_indep:.3f})')

# Add labels and title
plt.xlabel('k')
plt.ylabel('Density')
plt.margins(x=0.05, y=0)
plt.yticks([])
plt.ylim((0, 1))
plt.title(f'χ²-distribution (DF={dof_independ})')
plt.legend();

#### Directional interpretation


#### Chi-squared test with Pingouin


In [ ]:
import pingouin as pg

data_heart = pg.read_dataset('chi2_independence')

print("First rows and columns of the heart disease dataset:")
print(data_heart.iloc[:5,:5])

In [ ]:
# Perform chi-squared test using pingouin
expected_heart, observed_heart, stats_heart = pg.chi2_independence(
    data_heart,
    x='sex', y='target',
    correction=True,  # No Yates' correction for illustration
)

# Print the results
print("Observed frequencies (contingency table):")
print(observed_heart)
print()
print("Expected frequencies (under null hypothesis):")
print(expected_heart.round(2))
print()
print("χ² statistics:")
pg.print_table(stats_heart.round(4))

## Cochran-Armitage test for trend
### How the trend test works


### Example of varying the dose of a drug


#### Table vs. Table2×2
#### Cochran-Armitage test with statsmodels


#### Interpreting the results of the Cochran-Armitage test


In [ ]:
from statsmodels.stats.contingency_tables import Table

# Example of toxicology data: row = dose group
# columns = counts of animals with tumor vs. without tumor
tox_tumor_counts = np.array([
    [0, 50],  # Control group
    [1, 49],  # Low dose (1 animal with out of 50 total)
    [3, 47],  # Medium dose
    [6, 44]   # High dose
])

# Perform Cochran-Armitage trend test
trend_test_result = Table(
    tox_tumor_counts,
    shift_zeros=False
).test_ordinal_association(
    # row_scores=np.array([0, 1, 2, 3]),  # Default score
    col_scores=np.array([1, 0]),  # Reversed trend
)

print("Cochran-Armitage test for trend:")
print(trend_test_result)
print()

# Extract P value, and divide by 2 to obtain one-sided P value
p_value_trend_test = trend_test_result.pvalue / 2

print(f"P value (one-sided) = {p_value_trend_test:.4f}")

# Interpret results
if p_value_trend_test < α:
    print(
        "Reject the null hypothesis\n",
        "There is a significant increasing trend in tumor rates \
with increasing dose.")
else:
    print(
        "Fail to reject the null hypothesis\n",
        "There is not enough evidence to suggest a trend.")

#### Alternatives in R


## Case-control studies
### Statistical analysis in case-control studies


### Example of cholera vaccination


In [ ]:
# Create the contingency table from the cholera vaccine data
data_cholera = [
    [10, 94],
    [33, 78]]

table_cholera = Table2x2(data_cholera)

# Calculate the odds ratio (OR) and the 95% confidence interval
oddsratio_cholera = table_cholera.oddsratio
ci_cholera = np.array(table_cholera.oddsratio_confint())

print("Analysis of the cholera vaccination study:")
print(f" Odds ratio (OR) = {oddsratio_cholera:.2f}")
print(r" 95% CI (OR):",  ci_cholera.round(3))

In [ ]:
oddsratio_fisher_cholera, p_value_fisher_cholera=fisher_exact(data_cholera)

print("Analysis of the cholera vaccination study:")
print(f" Fisher's exact test P value = {p_value_fisher_cholera:.4f}")

### Isotretinoin and bowel disease


In [ ]:
# Create the contingency table from the isotretinoin data
data_isotretinoin = [
    [25, 213],
    [1935, 19216]]

table_isotretinoin = Table2x2(data_isotretinoin)

# Calculate the odds ratio (OR) and the 95% CI
oddsratio_isotretinoin = table_isotretinoin.oddsratio
ci_isotretinoin = np.array(table_isotretinoin.oddsratio_confint())

print("Analysis of the isotretinoin study:")
print(f" Odds ratio (OR) = {oddsratio_isotretinoin:.2f}")
print(r" 95% CI (OR):",  ci_isotretinoin.round(3))

In [ ]:
(
    chi2_isotretinoin,
    p_value_isotretinoin,
    dof_isotretinoin,
    expected_isotretinoin
) = chi2_contingency(data_isotretinoin, correction=False)

oddsratio_fisher_isotretinoin, p_value_fisher_isotretinoin = fisher_exact(
    data_isotretinoin)

print("Analysis of the isotretinoin study:")
print(f" χ² test P value = {p_value_isotretinoin:.3f}")
print(f" Fisher's exact test P value = {p_value_fisher_isotretinoin:.3f}")

## Conclusion
